# FlightGraph — Interactive Demonstration

**Airline Network Resilience & Disruption Simulator**

FlightGraph explores how graph analytics can identify critical airports,
evaluate network disruptions, and investigate alternative connections.

This notebook demonstrates the main capabilities of the application
using its FastAPI endpoints.

## Demonstration workflow

1. Verify that the application is running.
2. Explore airports and direct connections.
3. Find shortest routes and reachable airports.
4. Analyze the flight network using graph algorithms.
5. Simulate an airport closure.
6. Review the machine learning experiment.
7. Demonstrate the AI assistant.

**Dataset:** Historical OpenFlights airline network.

**Important:** FlightGraph does not provide real-time flight information.

In [1]:
"""Configure the FlightGraph demonstration."""

import httpx

BASE_URL = "http://localhost:8080"
TIMEOUT = 15.0

print("FlightGraph demonstration initialized.")


FlightGraph demonstration initialized.


## 1. Environment Verification

Before executing graph queries, verify that the FlightGraph API
is accessible through the Nginx reverse proxy.

The application uses three Docker services:

- Nginx: HTTP entry point.
- FastAPI: Application logic.
- Neo4j: Graph database and analytics.

In [ ]:
response = httpx.get(
    f"{BASE_URL}/health",
    timeout=TIMEOUT,
)

response.raise_for_status()

health = response.json()
print(health)

assert health["status"] == "ok", "FlightGraph API is not healthy."


{'status': 'ok'}


## 2. Network Exploration

FlightGraph represents airports as graph nodes and historical
flight connections as directed relationships.

We begin by retrieving information about Paris Charles de Gaulle
Airport (CDG).

In [ ]:
response = httpx.get(
    f"{BASE_URL}/api/airports/CDG",
    timeout=TIMEOUT,
)

response.raise_for_status()

airport = response.json()
airport


{'airport_id': 1382,
 'name': 'Charles de Gaulle International Airport',
 'city': 'Paris',
 'country': 'France',
 'iata': 'CDG'}

In [ ]:
response = httpx.get(
    f"{BASE_URL}/api/airports/CDG/connections",
    params={"limit": 10},
    timeout=TIMEOUT,
)

response.raise_for_status()

connections = response.json()

print(f"Retrieved {len(connections)} direct connections from CDG.")

connections[:5]


Retrieved 10 direct connections from CDG.


[{'iata': 'JFK',
  'name': 'John F Kennedy International Airport',
  'city': 'New York',
  'airline_count': 11},
 {'iata': 'ORD',
  'name': "Chicago O'Hare International Airport",
  'city': 'Chicago',
  'airline_count': 10},
 {'iata': 'MIA',
  'name': 'Miami International Airport',
  'city': 'Miami',
  'airline_count': 9},
 {'iata': 'TLS',
  'name': 'Toulouse-Blagnac Airport',
  'city': 'Toulouse',
  'airline_count': 8},
 {'iata': 'YUL',
  'name': 'Montreal / Pierre Elliott Trudeau International Airport',
  'city': 'Montreal',
  'airline_count': 6}]

## 3. Shortest Paths

FlightGraph finds shortest routes between airports using the directed
flight network.

A shortest route minimizes the number of flight connections, not
travel time, distance, or ticket price.

We investigate a route between Paris Charles de Gaulle (CDG)
and New York John F. Kennedy (JFK).

In [ ]:
response = httpx.get(
    f"{BASE_URL}/api/routes/shortest",
    params={
        "source": "CDG",
        "destination": "JFK",
    },
    timeout=TIMEOUT,
)

response.raise_for_status()

shortest_route = response.json()

shortest_route


{'airports': ['CDG', 'JFK'], 'flights': 1}

## 4. Multi-Hop Reachability

FlightGraph explores which destinations are reachable from a
departure airport within a maximum number of directed flight connections.

This example searches for airports reachable from CDG using
at most two flights.

The API limits the response to 20 destinations, so the result
is a sample of reachable airports rather than a complete count
of network accessibility.

In [ ]:
response = httpx.get(
    f"{BASE_URL}/api/airports/CDG/reachable",
    params={"max_hops": 2},
    timeout=TIMEOUT,
)

response.raise_for_status()

reachable_airports = response.json()

print(f"Retrieved {len(reachable_airports)} reachable airports.")

reachable_airports[:10]


Retrieved 20 reachable airports.


[{'iata': 'AAE', 'name': 'Rabah Bitat Airport', 'hops': 1},
 {'iata': 'ABJ', 'name': 'Port Bouet Airport', 'hops': 1},
 {'iata': 'ABV', 'name': 'Nnamdi Azikiwe International Airport', 'hops': 1},
 {'iata': 'ABZ', 'name': 'Aberdeen Dyce Airport', 'hops': 1},
 {'iata': 'ADB', 'name': 'Adnan Menderes International Airport', 'hops': 1},
 {'iata': 'ADD', 'name': 'Addis Ababa Bole International Airport', 'hops': 1},
 {'iata': 'AGA', 'name': 'Al Massira Airport', 'hops': 1},
 {'iata': 'AGP', 'name': 'Málaga Airport', 'hops': 1},
 {'iata': 'AJA', 'name': 'Ajaccio-Napoléon Bonaparte Airport', 'hops': 1},
 {'iata': 'ALG', 'name': 'Houari Boumediene Airport', 'hops': 1}]

## 5. Airport Disruption Simulation

FlightGraph evaluates how removing an airport affects the structure
of the historical airline network.

The simulator creates a temporary Neo4j GDS graph without the
selected airport, calculates connectivity metrics, and compares
the results with the original network.

The underlying Neo4j database is not permanently modified.

### Research question

How does closing Paris Charles de Gaulle Airport (CDG) affect
global airline network connectivity?

We evaluate:

- Number of airports and directed connections.
- Number of weakly connected components.
- Size of the largest connected component.
- Number of isolated airports.

**Important:** Weakly Connected Components (WCC) ignores the
direction of flight connections. It measures structural fragmentation,
not actual passenger reachability.

In [ ]:
response = httpx.get(
    f"{BASE_URL}/api/disruptions/airports/CDG",
    timeout=60.0,
)

print("HTTP status:", response.status_code)
response.raise_for_status()

disruption = response.json()
disruption


HTTP status: 200


{'closed_airport': {'airport_id': 1382,
  'name': 'Charles de Gaulle International Airport',
  'iata': 'CDG'},
 'lost_routes': 470,
 'baseline': {'total_components': 4491,
  'largest_component': 3188,
  'total_airports': 7698,
  'isolated_airports': 4484},
 'disrupted': {'total_components': 4492,
  'largest_component': 3186,
  'total_airports': 7697,
  'isolated_airports': 4485},
 'changes': {'components': 1, 'largest_component': -2, 'isolated_airports': 1}}

In [ ]:
import pandas as pd

baseline = disruption["baseline"]
disrupted = disruption["disrupted"]

metrics = {
    "Airports": ("total_airports",),
    "Connected components": ("total_components",),
    "Largest component": ("largest_component",),
    "Isolated airports": ("isolated_airports",),
}

rows = []

for label, (key,) in metrics.items():
    before = baseline[key]
    after = disrupted[key]

    rows.append({
        "Metric": label,
        "Before": before,
        "After": after,
        "Change": after - before,
    })

rows.append({
    "Metric": "Directed connections lost",
    "Before": None,
    "After": None,
    "Change": -disruption["lost_routes"],
})

comparison = pd.DataFrame(rows)
comparison


,Metric,Before,After,Change
0,Airports,7698.0,7697.0,-1
1,Connected components,4491.0,4492.0,1
2,Largest component,3188.0,3186.0,-2
3,Isolated airports,4484.0,4485.0,1
4,Directed connections lost,NaN,NaN,-470


### Interpretation

The CDG closure simulation removes a substantial number of
directed airport connections.

However, the largest weakly connected component decreases
only slightly, suggesting that much of the network remains
structurally connected.

This illustrates an important distinction between airport
connectivity and network resilience.

An airport can serve many connections without being a
critical articulation point for weak connectivity.

**Limitations:** The simulation does not model passenger
demand, flight schedules, cancellations, or operational
rerouting.

## 6. Graph Analytics

FlightGraph uses Neo4j Graph Data Science (GDS) to analyze
the structure of the historical airline network.

Two algorithms are implemented:

**PageRank**
- Measures the structural importance of airports.
- Considers incoming connections from other airports.
- Does not directly measure the consequences of an airport closure.

**Weakly Connected Components (WCC)**
- Identifies connected groups of airports.
- Ignores relationship direction.
- Helps evaluate structural fragmentation.

The disruption simulator uses WCC to compare the network
before and after an airport closure.

In [ ]:
print("Baseline network connectivity")
print("-" * 35)

print(f"Airports: {baseline['total_airports']:,}")
print(f"Connected components: {baseline['total_components']:,}")
print(f"Largest component: {baseline['largest_component']:,}")
print(f"Isolated airports: {baseline['isolated_airports']:,}")


Baseline network connectivity
-----------------------------------
Airports: 7,698
Connected components: 4,491
Largest component: 3,188
Isolated airports: 4,484


## 7. Machine Learning — Route Link Prediction

FlightGraph includes a supervised learning experiment
investigating whether graph-based features improve the
identification of historical airline connections.

### Models

**Baseline:** Logistic Regression using geographic distance.

**Graph-enhanced:** Logistic Regression using:

- Geographic distance.
- Source airport out-degree.
- Destination airport in-degree.
- Common neighbors.

### Previously Recorded Results

| Metric | Distance baseline | Graph-enhanced |
|---|---:|---:|
| Initial ROC-AUC | 0.9357 | 0.9922 |
| Initial F1-score | 0.8748 | 0.9685 |
| Distance-restricted ROC-AUC | 0.7801 | 0.9531 |
| Distance-restricted F1-score | 0.6667 | 0.8696 |

These are results from the existing ML experiment,
not measurements generated by this demonstration.

### Limitations

The experiment uses historical OpenFlights data.

Unrecorded routes are treated as negative examples,
which does not establish that they are impossible routes.

Evaluation results depend on the negative-sampling strategy
and do not represent real-world route probabilities.

The distance-restricted experiment is exploratory rather
than an independent validation.

The full methodology is available in
`notebooks/02_route_link_prediction.ipynb`.

## 8. AI Assistant

FlightGraph includes an AI assistant that interprets
natural-language questions about the airline network.

Instead of executing unrestricted LLM-generated Cypher,
the assistant maps questions to approved graph operations.

This design prioritizes validation, predictable execution,
and evidence-grounded responses.

The demonstration below is optional because it requires
a configured OpenAI API key.

In [ ]:
RUN_AI_DEMO = False

if RUN_AI_DEMO:
    response = httpx.post(
        f"{BASE_URL}/api/assistant/ask",
        json={
            "question": "What are the direct connections from CDG?"
        },
        timeout=60.0,
    )

    response.raise_for_status()
    assistant_result = response.json()

    display(assistant_result)
else:
    print("AI assistant demonstration skipped.")
    print("Set RUN_AI_DEMO = True to enable it.")


AI assistant demonstration skipped.
Set RUN_AI_DEMO = True to enable it.


## 9. Conclusion

FlightGraph demonstrates how a graph database can support
airline network exploration, connectivity analysis,
and airport disruption simulation.

### Main Findings

- Neo4j enables directed airport-to-airport traversal.
- Graph algorithms help describe network structure.
- Airport closures can remove many connections without
  necessarily causing substantial weak-connectivity fragmentation.
- Graph-based features improve historical route identification
  in the documented ML experiment.
- A restricted AI assistant enables natural-language access
  to selected graph operations.

### Limitations

FlightGraph uses historical airline route data rather than
real-time schedules.

Connectivity metrics do not capture passenger demand,
operational constraints, or cascading flight cancellations.

The application is intended for educational graph analytics
and software engineering experimentation.